# Phase 6 (Abstractive Summarization) - Model 1 (GQA) + LoRMA (perm)
Chunk-level summarization with the Phase-1 Model 1 decoder. LoRMA is written by hand (no peft).

Training: `[chunk] SEP [chunk target] SEP`, loss only on the target. Inference: summarize every chunk of a document, join the pieces in order.

**Attach (Kaggle "Add Input"):**
1. The Phase-6 data-prep output (`tokenized_summ.pkl`, `aligned_summ.pkl`, `summ_utils.py`)
2. Your Phase-1 Model 1 folder (`legal-gqa-final`, contains `config.json` and weights)
3. The file `legal_gqa_model.py`

**Output:** `/kaggle/working/summ_m1_lorma/` (LoRMA weights, validation predictions, `prediction.json`, `submission.zip`)

## 1. Configuration

In [ ]:
import os
import sys
import json
import glob
import math
import time
import random
import shutil
import pickle
import zipfile

import numpy as np

# Env vars are only for local testing. On Kaggle leave them alone.
DATA_DIR = os.environ.get("SUM_DATA_DIR", None)
MODEL_DIR = os.environ.get("SUM_MODEL_DIR", None)
MODEL_PY = os.environ.get("SUM_MODEL_PY", None)
OUT_DIR = os.environ.get("SUM_OUT_DIR", "/kaggle/working/summ_m1_lorma")

MODEL_NAME_HINT = "legal-gqa"     # folder name fragment of the Phase-1 Model 1 folder
MODEL_PY_NAME = "legal_gqa_model.py"

SMOKE_TEST = False                # True = tiny run to check the code end to end
if os.environ.get("SUM_SMOKE") == "1":
    SMOKE_TEST = True
SMOKE_DOCS = 24
SMOKE_EVAL_DOCS = 3
SMOKE_TEST_DOCS = 3

SEED = 42
VAL_FOLD = 0                      # fold 0 is held out for validation (train on the other 4 folds)
EVAL_DOCS = 60                    # validation docs that get a full generated summary

# What to run
RUN_TRAIN = True
RUN_VAL = True
RUN_TEST = True
RESUME = True                     # pick up a checkpoint from an attached earlier run

# LoRMA (hand-written), on every linear layer inside the transformer blocks
LORMA_VARIANT = "perm"             # "perm" (LoRMA-pi: dense d x d rotation per layer, slower) or "plus" (LoRMA+, cheap)
LORMA_R = 16
LORMA_ALPHA = 16                  # only used by plus (scale = alpha / r)
LORMA_DROPOUT = 0.05              # only used by plus

# Training
LR = 5e-4
WEIGHT_DECAY = 0.0
EPOCHS = 2
WARMUP_FRAC = 0.03
MICRO_BATCH = 4                   # chunk examples per forward pass
GRAD_ACC = 4                      # effective batch = 16 examples
GRAD_CLIP = 1.0
EMPTY_KEEP_PROB = 0.5             # share of empty-target chunks kept in training (they teach "say nothing")
USE_CHECKPOINT = True             # gradient checkpointing, needed on a 16 GB T4
SAVE_EVERY_STEPS = 200
LOG_EVERY_STEPS = 20
SESSION_CAP_HOURS = 10.5          # the run saves and stops, and resumes on the next run

# Generation
GEN_BATCH = 8                     # chunks per generation batch (prefill builds full logits, keep it small)
LEN_MULT = 0.6                    # max new tokens per batch = LEN_MULT x the longest prompt (capped by the target budget)
REP_PENALTY = 1.0                 # 1.0 = off
ROUGE_L_MAX_WORDS = 1500          # ROUGE-L is computed on the first N words only (keeps the LCS fast)
USE_BERTSCORE = False             # needs the bert_score package and a downloaded scorer model; evaluation only

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
if SMOKE_TEST:
    EPOCHS = 1
    print("SMOKE_TEST is on: tiny data, 1 epoch, to check the code only.")
print("Config loaded.")

## 2. Imports and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint

START_TIME = time.time()
SESSION_CAP_SEC = SESSION_CAP_HOURS * 3600
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    if torch.cuda.is_bf16_supported():
        AMP_DTYPE = torch.bfloat16
        USE_SCALER = False
    else:
        AMP_DTYPE = torch.float16
        USE_SCALER = True
else:
    AMP_DTYPE = torch.float32
    USE_SCALER = False
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE, "| amp dtype:", AMP_DTYPE, "| grad scaler:", USE_SCALER)

## 3. Find the inputs

In [ ]:
def find_one_file(search_roots, file_name):
    for root in search_roots:
        for dir_path, dir_names, file_names in os.walk(root):
            if file_name in file_names:
                return os.path.join(dir_path, file_name)
    return None


def find_model_dir(name_hint):
    for dir_path, dir_names, file_names in os.walk("/kaggle/input"):
        low = dir_path.lower()
        if "config.json" in file_names and name_hint in low and "checkpoint" not in low:
            return dir_path
    return None


if DATA_DIR is None:
    pkl_path = find_one_file(["/kaggle/input", "/kaggle/working"], "tokenized_summ.pkl")
    if pkl_path is None:
        raise FileNotFoundError("tokenized_summ.pkl not found. Attach the Phase-6 data-prep output.")
    DATA_DIR = os.path.dirname(pkl_path)

if MODEL_DIR is None:
    MODEL_DIR = find_model_dir(MODEL_NAME_HINT)
    if MODEL_DIR is None:
        raise FileNotFoundError("No Phase-1 Model 1 folder with config.json found. Attach legal-gqa-final.")

if MODEL_PY is None:
    MODEL_PY = find_one_file(["/kaggle/input", "/kaggle/working"], MODEL_PY_NAME)
    if MODEL_PY is None:
        raise FileNotFoundError("legal_gqa_model.py not found. Attach it (see the header).")

print("data dir:", DATA_DIR)
print("model dir:", MODEL_DIR)
print("model file:", MODEL_PY)

## 4. Load the data and build the chunk examples

In [ ]:
for name in ["summ_utils.py"]:
    if os.path.exists(os.path.join(DATA_DIR, name)):
        shutil.copy(os.path.join(DATA_DIR, name), name)
sys.path.insert(0, os.getcwd())
import summ_utils

with open(os.path.join(DATA_DIR, "tokenized_summ.pkl"), "rb") as f:
    DATA = pickle.load(f)
ALIGNED = summ_utils.load_aligned(os.path.join(DATA_DIR, "aligned_summ.pkl"))

META = DATA["meta"]
SEP_ID = int(META["sep_id"])
PAD_ID = int(META["pad_id"])
MODEL_CTX = int(META["model_ctx"])
CHUNK_DOC_LEN = int(META["chunk_doc_len"])
CHUNK_SUM_BUDGET = int(META["chunk_sum_budget"])
print("SEP:", SEP_ID, "| PAD:", PAD_ID, "| ctx:", MODEL_CTX, "| chunk doc:", CHUNK_DOC_LEN, "| target budget:", CHUNK_SUM_BUDGET)

# Pool = train + dev. Validation = the docs of VAL_FOLD.
TRAIN_ROWS = []
VAL_ROWS = []
for split in ["train", "dev"]:
    for row in DATA[split]:
        row["split"] = split
        if row["fold"] == VAL_FOLD:
            VAL_ROWS.append(row)
        else:
            TRAIN_ROWS.append(row)
TEST_ROWS = DATA["test"]

rng_docs = np.random.RandomState(SEED)
val_order = rng_docs.permutation(len(VAL_ROWS))
VAL_ROWS = [VAL_ROWS[i] for i in val_order]

if SMOKE_TEST:
    TRAIN_ROWS = TRAIN_ROWS[:SMOKE_DOCS]
    VAL_ROWS = VAL_ROWS[:SMOKE_EVAL_DOCS]
    TEST_ROWS = TEST_ROWS[:SMOKE_TEST_DOCS]
else:
    VAL_ROWS = VAL_ROWS[:EVAL_DOCS]

TRAIN_ALIGNED = [ALIGNED[row["split"]][row["id"]] for row in TRAIN_ROWS]

# One training example per (document, chunk). Empty targets are thinned out.
rng_pairs = np.random.RandomState(SEED)
PAIRS = []
PAIR_LEN = []
n_empty_seen = 0
n_empty_kept = 0
for ri in range(len(TRAIN_ROWS)):
    entry = TRAIN_ALIGNED[ri]
    bounds = entry["bounds"]
    for k in range(len(entry["targets"])):
        target_len = len(entry["targets"][k])
        if target_len == 0:
            n_empty_seen += 1
            if rng_pairs.rand() > EMPTY_KEEP_PROB:
                continue
            n_empty_kept += 1
        chunk_len = int(bounds[k + 1] - bounds[k])
        PAIRS.append((ri, k))
        PAIR_LEN.append(chunk_len + min(target_len, CHUNK_SUM_BUDGET))

print("train docs:", len(TRAIN_ROWS), "| val docs:", len(VAL_ROWS), "| test docs:", len(TEST_ROWS))
print("training examples:", len(PAIRS), "| empty targets kept:", n_empty_kept, "of", n_empty_seen)
print("example length: mean %.0f | max %d" % (np.mean(PAIR_LEN), max(PAIR_LEN)))


def make_batch(pair_list):
    seqs = []
    labs = []
    for ri, k in pair_list:
        row = TRAIN_ROWS[ri]
        entry = TRAIN_ALIGNED[ri]
        b = entry["bounds"]
        chunk = row["doc_ids"][b[k]:b[k + 1]]
        target = entry["targets"][k]
        inp, lab = summ_utils.build_chunk_example(chunk, target, SEP_ID, CHUNK_DOC_LEN, CHUNK_SUM_BUDGET)
        seqs.append(inp)
        labs.append(lab)
    T = max(len(s) for s in seqs)
    ids = np.full((len(seqs), T), PAD_ID, dtype=np.int64)
    labels = np.full((len(seqs), T), -100, dtype=np.int64)
    for i in range(len(seqs)):
        ids[i, :len(seqs[i])] = seqs[i]
        labels[i, :len(labs[i])] = labs[i]
    n_tok = int((labels[:, 1:] != -100).sum())
    return torch.from_numpy(ids), torch.from_numpy(labels), n_tok


def epoch_batches(epoch):
    rng = np.random.RandomState(SEED + epoch)
    order = rng.permutation(len(PAIRS))
    group = MICRO_BATCH * 50
    batches = []
    for g in range(0, len(order), group):
        idx = sorted(order[g:g + group].tolist(), key=lambda i: PAIR_LEN[i])
        for s in range(0, len(idx), MICRO_BATCH):
            batches.append([PAIRS[i] for i in idx[s:s + MICRO_BATCH]])
    rng.shuffle(batches)
    return batches


from transformers import AutoTokenizer
tok_dir = META["tokenizer_dir"]
if not os.path.exists(os.path.join(tok_dir, "tokenizer.json")):
    found = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
    if len(found) == 0:
        raise FileNotFoundError("Tokenizer folder from the data-prep run not found. Attach the tokenizer dataset.")
    tok_dir = os.path.dirname(sorted(found)[0])
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
print("tokenizer:", tok_dir)

## 5. Load the frozen Phase-1 Model 1

In [ ]:
import importlib.util
import inspect

sys.path.insert(0, os.path.dirname(MODEL_PY))
spec = importlib.util.spec_from_file_location("legal_gqa_model", MODEL_PY)
legal_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(legal_mod)

LM, load_info = legal_mod.LegalGQAForCausalLM.from_pretrained(MODEL_DIR, output_loading_info=True)
missing_keys = []
for key in load_info.get("missing_keys", []):
    if key != "lm_head.weight":          # tied to the token embedding
        missing_keys.append(key)
unexpected_keys = list(load_info.get("unexpected_keys", []))
print("Phase-1 weights loaded | missing keys:", len(missing_keys), "| unexpected keys:", len(unexpected_keys))
if len(missing_keys) > 0:
    raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing_keys[:5]))
if LM.config.vocab_size != META["vocab_size"]:
    print("WARNING: model vocab", LM.config.vocab_size, "!= tokenizer vocab", META["vocab_size"])
if LM.config.vocab_size <= SEP_ID:
    raise RuntimeError("tokenizer / model vocab mismatch")

LM.float()
LM = LM.to(DEVICE)
LM.eval()
for p in LM.parameters():
    p.requires_grad = False

N_BASE = sum(p.numel() for p in LM.parameters())
print("model:", type(LM).__name__, "| params: %.2fM" % (N_BASE / 1e6))
print("forward arguments:", list(inspect.signature(LM.forward).parameters.keys()))


def backbone_hidden(input_ids):
    # Same computation as the Phase-1 forward(), stopped before lm_head. Causal model, right padding is harmless.
    T = input_ids.shape[1]
    x = LM.drop(LM.wte(input_ids))
    cos, sin = LM._rope_slice(0, T, input_ids.device, x.dtype)
    for blk in LM.blocks:
        if torch.is_grad_enabled() and USE_CHECKPOINT:
            def run(h, blk=blk):
                return blk(h, cos, sin, None, True, None, False)[0]
            x = torch_checkpoint(run, x, use_reentrant=False)
        else:
            x = blk(x, cos, sin, None, True, None, False)[0]
    return LM.ln_f(x)


# My manual forward must agree with the model's own forward.
with torch.no_grad():
    probe = torch.tensor(np.random.randint(3, LM.config.vocab_size, size=(1, 48)), device=DEVICE)
    own_logits = LM.lm_head(backbone_hidden(probe)).float()
    ref_logits = LM(input_ids=probe).logits.float()
    agree = float((own_logits.argmax(-1) == ref_logits.argmax(-1)).float().mean())
    diff = float((own_logits - ref_logits).abs().max())
print("manual forward vs model forward: top-1 agreement %.3f | max logit diff %.4f" % (agree, diff))
assert agree > 0.9, "manual forward does not match the model forward"

## 6. Hand-written LoRMA

LoRMA (Low-Rank Multiplicative Adaptation): the frozen weight is multiplied from the output side, not added to. `perm`: `W = P(BA) W0` with row `i` of `BA` rotated by `i` places (starts at the identity). `plus`: `W = ((alpha/r) BA + I) W0`. Both start exactly at the Phase-1 weights.

In [ ]:
_ROT_CACHE = {}
_EYE_CACHE = {}


def hand_dropout(x, p, training):
    # dropout written by hand: zero some entries, scale the rest
    if (not training) or p == 0.0:
        return x
    keep_prob = 1.0 - p
    rand = torch.rand_like(x)
    mask = (rand < keep_prob).to(x.dtype)
    return x * mask / keep_prob


def hand_uniform_(tensor, bound):
    # fill a tensor with numbers drawn uniformly from [-bound, bound]
    with torch.no_grad():
        tensor.copy_(torch.rand_like(tensor) * 2.0 * bound - bound)


def get_rot_idx(d, device):
    # index table for the row rotation: entry (i, j) = (j - i) mod d
    # shared by all layers with the same output size
    key = (d, str(device))
    if key not in _ROT_CACHE:
        rows = torch.arange(d, device=device).unsqueeze(1)
        cols = torch.arange(d, device=device).unsqueeze(0)
        diff = cols - rows
        _ROT_CACHE[key] = diff % d
    return _ROT_CACHE[key]


def get_eye(d, device):
    # identity matrix built by hand: zeros with ones on the diagonal
    key = (d, str(device))
    if key not in _EYE_CACHE:
        eye = torch.zeros(d, d, device=device)
        diag = torch.arange(d, device=device)
        eye[diag, diag] = 1.0
        _EYE_CACHE[key] = eye
    return _EYE_CACHE[key]


def rotate_rows(M):
    # row i of M is shifted right by i places (wrapping around). Written with plain index tensors.
    d = M.shape[0]
    rows = torch.arange(d, device=M.device).unsqueeze(1).expand(d, d)
    cols = get_rot_idx(d, M.device)
    return M[rows, cols]


class LoRMALinear(nn.Module):
    # LoRMA: multiplicative update on the output side of the frozen layer. W0 is frozen, z = W0 x.
    #   plus: W = ((alpha / r) B A + I) W0      ->  y = z + (alpha / r) B (A z)
    #   perm: W = P(B A) W0                     ->  y = R z, R = row i of (B A) rotated by i places
    # Both start exactly at the Phase-1 weights (plus: B = 0; perm: R = I).
    # perm is computed as y = z + z (R - I)^T so the small learned part is not lost in fp16.
    def __init__(self, base, r, alpha, dropout, variant):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base.weight.shape[0]
        self.d = d_out
        self.r = r
        self.variant = variant
        self.scale = alpha / r
        self.p_drop = dropout
        dev = base.weight.device
        self.lorma_A = nn.Parameter(torch.empty(r, d_out, device=dev, dtype=torch.float32))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r, device=dev, dtype=torch.float32))
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1.0 / math.sqrt(self.d)
        hand_uniform_(self.lorma_A, bound)
        with torch.no_grad():
            self.lorma_B.zero_()
            if self.variant == "perm":
                self.lorma_B[:, 0] = 1.0       # B A then has a single column of ones ...
                self.lorma_A[0].zero_()
                self.lorma_A[0, 0] = 1.0       # ... and the row rotation turns it into the identity

    def forward(self, x):
        z = self.base(x)
        if self.variant == "plus":
            z32 = z.float()
            z32 = hand_dropout(z32, self.p_drop, self.training)
            low = z32 @ self.lorma_A.t()
            delta = low @ self.lorma_B.t()
            delta = delta * self.scale
            return z + delta.to(z.dtype)
        # perm: build R from B A, then add z (R - I)^T
        with torch.autocast(device_type=z.device.type, enabled=False):
            M = self.lorma_B @ self.lorma_A
            R = rotate_rows(M)
            D = R - get_eye(self.d, M.device)
        return z + z @ D.t()


def set_submodule(root, dotted_name, new_module):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


# Targets: every linear layer inside the blocks. Not the embedding, not lm_head.
LORMA_TARGETS = []
for name, module in LM.named_modules():
    if isinstance(module, nn.Linear) and name.startswith("blocks."):
        LORMA_TARGETS.append(name)
if len(LORMA_TARGETS) == 0:
    raise RuntimeError("No linear layers found under blocks. Check the Model 1 module names.")
print("LoRMA variant:", LORMA_VARIANT, "| target layers:", len(LORMA_TARGETS), "| first few:", LORMA_TARGETS[:5])

with torch.no_grad():
    check_ids = torch.tensor([[5, 9, 12, 40, 77, SEP_ID]], device=DEVICE)
    before = backbone_hidden(check_ids).float().clone()

LORMA_MODULES = []
for name in LORMA_TARGETS:
    wrapped = LoRMALinear(LM.get_submodule(name), LORMA_R, LORMA_ALPHA, LORMA_DROPOUT, LORMA_VARIANT)
    set_submodule(LM, name, wrapped)
    LORMA_MODULES.append(wrapped)

with torch.no_grad():
    after = backbone_hidden(check_ids).float()
identity_diff = float((before - after).abs().max())
print("identity check (max abs diff at init, should be 0):", identity_diff)
assert identity_diff < 1e-4, "LoRMA changed the Phase-1 output at init"

LORMA_PARAMS = []
for m in LORMA_MODULES:
    LORMA_PARAMS.append(m.lorma_A)
    LORMA_PARAMS.append(m.lorma_B)
N_LORMA = sum(p.numel() for p in LORMA_PARAMS)
print("trainable LoRMA params: %s (%.2f%% of the model)" % (f"{N_LORMA:,}", 100.0 * N_LORMA / N_BASE))


def set_train_mode():
    LM.eval()                      # Phase-1 dropout stays off
    for m in LORMA_MODULES:
        m.train()                  # LoRMA dropout on


def set_eval_mode():
    LM.eval()
    for m in LORMA_MODULES:
        m.eval()


def lorma_state():
    return {name: (m.lorma_A.detach().cpu(), m.lorma_B.detach().cpu()) for name, m in zip(LORMA_TARGETS, LORMA_MODULES)}


def load_lorma_state(state):
    for name, m in zip(LORMA_TARGETS, LORMA_MODULES):
        a, b = state[name]
        with torch.no_grad():
            m.lorma_A.copy_(a.to(m.lorma_A.device))
            m.lorma_B.copy_(b.to(m.lorma_B.device))

## 7. Training

In [ ]:
def lm_loss_sum(input_ids, labels):
    # Sum of token losses over the supervised (summary) positions. Logits only where needed.
    input_ids = input_ids.to(DEVICE)
    labels = labels.to(DEVICE)
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        hidden = backbone_hidden(input_ids)
    shifted_hidden = hidden[:, :-1]
    target = labels[:, 1:]
    selected = target != -100
    h_sel = shifted_hidden[selected]
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        logits = LM.lm_head(h_sel)
    return F.cross_entropy(logits.float(), target[selected], reduction="sum")


CKPT_PATH = os.path.join(OUT_DIR, "ckpt_lorma.pt")
LORMA_PATH = os.path.join(OUT_DIR, "lorma_weights.pt")
STOPPED_EARLY = False
TRAIN_SECONDS = 0.0
LAST_LOSS = None

if RUN_TRAIN and len(PAIRS) > 0:
    micro_per_epoch = int(math.ceil(len(PAIRS) / MICRO_BATCH))
    steps_per_epoch = int(math.ceil(micro_per_epoch / GRAD_ACC))
    total_steps = EPOCHS * steps_per_epoch
    warm_steps = max(1, int(WARMUP_FRAC * total_steps))
    print("micro-batches per epoch:", micro_per_epoch, "| optimizer steps per epoch:", steps_per_epoch, "| total steps:", total_steps)

    optimizer = torch.optim.AdamW(LORMA_PARAMS, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    def lr_at(step):
        if step < warm_steps:
            return LR * (step + 1) / warm_steps
        progress = (step - warm_steps) / max(1, total_steps - warm_steps)
        return LR * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

    state = {"epoch": 0, "micro": 0, "step": 0, "seconds": 0.0}

    if RESUME:
        found = None
        if os.path.exists(CKPT_PATH):
            found = CKPT_PATH
        else:
            found = find_one_file(["/kaggle/input"], "ckpt_lorma.pt")
        if found is not None:
            ck = torch.load(found, map_location="cpu", weights_only=False)
            load_lorma_state(ck["lorma"])
            optimizer.load_state_dict(ck["opt"])
            if USE_SCALER and ck.get("scaler") is not None:
                scaler.load_state_dict(ck["scaler"])
            state = ck["state"]
            print("resumed from", found, "| epoch", state["epoch"], "| micro", state["micro"], "| step", state["step"])
            if found != CKPT_PATH:
                shutil.copy(found, CKPT_PATH)

    def save_ckpt():
        torch.save({"lorma": lorma_state(), "opt": optimizer.state_dict(),
                    "scaler": scaler.state_dict() if USE_SCALER else None, "state": state}, CKPT_PATH)

    set_train_mode()
    run_loss = 0.0
    run_tok = 0
    t_run = time.time()
    t_last_log = time.time()
    seconds_before = state["seconds"]

    for epoch in range(state["epoch"], EPOCHS):
        batches = epoch_batches(epoch)
        start_micro = state["micro"] if epoch == state["epoch"] else 0
        micro = start_micro
        while micro < len(batches):
            group = batches[micro:micro + GRAD_ACC]
            built = [make_batch(b) for b in group]
            total_tokens = max(1, sum(x[2] for x in built))

            for g in optimizer.param_groups:
                g["lr"] = lr_at(state["step"])

            group_loss = 0.0
            for ids, labels, n_tok in built:
                loss_sum = lm_loss_sum(ids, labels)
                group_loss += float(loss_sum.detach())
                scaler.scale(loss_sum / total_tokens).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(LORMA_PARAMS, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            micro += len(group)
            state["epoch"] = epoch
            state["micro"] = micro
            state["step"] += 1
            run_loss += group_loss
            run_tok += total_tokens
            LAST_LOSS = group_loss / total_tokens

            if state["step"] % LOG_EVERY_STEPS == 0:
                elapsed = time.time() - t_last_log
                per_step = elapsed / LOG_EVERY_STEPS
                left = total_steps - state["step"]
                print("epoch %d | step %d/%d | loss %.4f | lr %.2e | %.1f s/step | eta %.0f min" % (
                    epoch, state["step"], total_steps, run_loss / max(run_tok, 1), lr_at(state["step"]),
                    per_step, left * per_step / 60.0))
                run_loss = 0.0
                run_tok = 0
                t_last_log = time.time()

            state["seconds"] = seconds_before + (time.time() - t_run)
            if state["step"] % SAVE_EVERY_STEPS == 0:
                save_ckpt()
            if time.time() - START_TIME > SESSION_CAP_SEC:
                save_ckpt()
                STOPPED_EARLY = True
                print("Session time cap reached. Checkpoint saved. Attach this notebook's output as an input and run again to resume.")
                break
        if STOPPED_EARLY:
            break
        state["epoch"] = epoch + 1
        state["micro"] = 0
        save_ckpt()
        print("epoch", epoch, "finished")

    TRAIN_SECONDS = state["seconds"]
    if not STOPPED_EARLY:
        state["epoch"] = EPOCHS
        save_ckpt()
        torch.save(lorma_state(), LORMA_PATH)
        print("training done | %.1f min | last loss %.4f" % (TRAIN_SECONDS / 60.0, LAST_LOSS if LAST_LOSS is not None else float("nan")))
else:
    if os.path.exists(LORMA_PATH):
        load_lorma_state(torch.load(LORMA_PATH, map_location="cpu"))
        print("loaded LoRMA weights from", LORMA_PATH)
    else:
        found = find_one_file(["/kaggle/input"], "lorma_weights.pt")
        if found is not None:
            load_lorma_state(torch.load(found, map_location="cpu"))
            print("loaded LoRMA weights from", found)
        else:
            print("No training and no saved LoRMA weights: the model is the plain Phase-1 model.")

## 8. Generation (chunk by chunk, KV cache, greedy)

Own decoding loop over the model's forward (left padding for batches). First a check that padding and the cache do not change the logits.

In [ ]:
@torch.no_grad()
def generate_batch(prompts, max_new):
    # prompts: list of 1-D int arrays, each ending with SEP. Greedy decoding until SEP. Returns a list of id lists.
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((B, L), dtype=torch.long)
    for i, p in enumerate(prompts):
        ids[i, L - len(p):] = torch.as_tensor(np.asarray(p, dtype=np.int64))
        mask[i, L - len(p):] = 1
    ids = ids.to(DEVICE)
    mask = mask.to(DEVICE)

    out = LM(input_ids=ids, attention_mask=mask, use_cache=True)
    past = out.past_key_values
    logits = out.logits[:, -1].float()
    del out

    generated = torch.zeros((B, max_new), dtype=torch.long, device=DEVICE)
    finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    n_steps = 0
    for step in range(max_new):
        if REP_PENALTY != 1.0 and step > 0:
            prev = generated[:, :step]
            score = logits.gather(1, prev)
            score = torch.where(score < 0, score * REP_PENALTY, score / REP_PENALTY)
            logits = logits.scatter(1, prev, score)
        nxt = logits.argmax(dim=-1)
        nxt = torch.where(finished, torch.full_like(nxt, SEP_ID), nxt)
        generated[:, step] = nxt
        finished = finished | (nxt == SEP_ID)
        n_steps = step + 1
        if bool(finished.all()):
            break
        mask = torch.cat([mask, torch.ones((B, 1), dtype=torch.long, device=DEVICE)], dim=1)
        out = LM(input_ids=nxt.unsqueeze(1), attention_mask=mask, past_key_values=past, use_cache=True)
        past = out.past_key_values
        logits = out.logits[:, -1].float()
        del out

    results = []
    gen_cpu = generated[:, :n_steps].cpu().numpy()
    for i in range(B):
        seq = gen_cpu[i].tolist()
        if SEP_ID in seq:
            seq = seq[:seq.index(SEP_ID)]
        results.append(seq)
    return results


set_eval_mode()
LM.config.use_cache = True

# Check 1: left padding must not change the next-token logits. Check 2: the KV cache must match a full recompute.
probe_row = TRAIN_ROWS[0]
probe_ids = np.asarray(probe_row["doc_ids"][:120]).astype(np.int64)
probe_ids = np.concatenate([probe_ids, [SEP_ID]])
with torch.no_grad():
    single = torch.as_tensor(probe_ids, device=DEVICE).unsqueeze(0)
    ref = LM(input_ids=single).logits[0, -1].float()

    pad_n = 9
    padded = torch.cat([torch.full((1, pad_n), PAD_ID, dtype=torch.long, device=DEVICE), single], dim=1)
    pmask = torch.cat([torch.zeros((1, pad_n), dtype=torch.long, device=DEVICE), torch.ones_like(single)], dim=1)
    pad_logits = LM(input_ids=padded, attention_mask=pmask).logits[0, -1].float()
    pad_diff = float((ref - pad_logits).abs().max())

    o1 = LM(input_ids=single, use_cache=True)
    nxt_tok = o1.logits[:, -1].argmax(-1, keepdim=True)
    o2 = LM(input_ids=nxt_tok, past_key_values=o1.past_key_values, use_cache=True)
    full_next = LM(input_ids=torch.cat([single, nxt_tok], dim=1)).logits[0, -1].float()
    cache_diff = float((o2.logits[0, -1].float() - full_next).abs().max())

print("left-padding max logit diff: %.4f | KV-cache max logit diff: %.4f" % (pad_diff, cache_diff))
print("(small numbers are fine; fp16 gives differences of a few hundredths)")
assert pad_diff < 2.0, "left padding changes the logits a lot: the attention-mask path is not behaving"
assert cache_diff < 2.0, "KV cache disagrees with a full recompute: check how the model returns past_key_values"

t_probe = time.time()
test_out = generate_batch([probe_ids, probe_ids[:60].copy()], 8)
print("generation smoke:", [len(x) for x in test_out], "tokens in %.2f s" % (time.time() - t_probe))

In [ ]:
GEN_STATS = {"tokens": 0, "seconds": 0.0}


def summarize_documents(rows, label):
    # Every chunk of every document is summarized; pieces are joined in order.
    jobs = []
    for d, row in enumerate(rows):
        entry = ALIGNED[row["split"]][row["id"]] if "split" in row else ALIGNED["test"][row["id"]]
        pieces = summ_utils.chunk_slices(row["doc_ids"], entry["bounds"])
        for k, piece in enumerate(pieces):
            prompt = summ_utils.build_chunk_prompt(piece, SEP_ID, CHUNK_DOC_LEN)
            jobs.append((d, k, prompt))
    print(label, "| documents:", len(rows), "| chunks to summarize:", len(jobs))
    jobs.sort(key=lambda j: -len(j[2]))

    texts = {}
    t0 = time.time()
    total_new = 0
    for start in range(0, len(jobs), GEN_BATCH):
        batch = jobs[start:start + GEN_BATCH]
        longest = max(len(j[2]) for j in batch)
        max_new = int(min(CHUNK_SUM_BUDGET, max(16, LEN_MULT * longest)))
        outs = generate_batch([j[2] for j in batch], max_new)
        for (d, k, prompt), seq in zip(batch, outs):
            texts[(d, k)] = tokenizer.decode(seq, clean_up_tokenization_spaces=False) if len(seq) > 0 else ""
            total_new += len(seq)
        if (start // GEN_BATCH) % 25 == 0:
            print("  generated", min(start + GEN_BATCH, len(jobs)), "/", len(jobs), "| %.0f s" % (time.time() - t0))
    seconds = time.time() - t0
    GEN_STATS["tokens"] += total_new
    GEN_STATS["seconds"] += seconds
    print(label, "done | %d new tokens | %.1f s | %.1f tokens/s" % (total_new, seconds, total_new / max(seconds, 1e-6)))

    summaries = []
    for d, row in enumerate(rows):
        entry = ALIGNED[row["split"]][row["id"]] if "split" in row else ALIGNED["test"][row["id"]]
        n_chunks = len(entry["bounds"]) - 1
        pieces = [texts[(d, k)] for k in range(n_chunks)]
        summaries.append(summ_utils.join_chunk_summaries(pieces))
    return summaries

## 9. Validation (held-out fold)

In [ ]:
VAL_RESULT = {}


def rouge_l_capped(pred, ref):
    p = " ".join(pred.split()[:ROUGE_L_MAX_WORDS])
    r = " ".join(ref.split()[:ROUGE_L_MAX_WORDS])
    return summ_utils.rougeL_f1(p, r)


if STOPPED_EARLY:
    print("Training stopped early, skipping validation. Run again to resume.")
elif RUN_VAL and len(VAL_ROWS) > 0:
    val_summaries = summarize_documents(VAL_ROWS, "validation")
    r1 = []
    rl = []
    r1_lead = []
    rep = []
    ratio = []
    for row, pred in zip(VAL_ROWS, val_summaries):
        ref = row["summary"]
        r1.append(summ_utils.rouge1_f1(pred, ref))
        rl.append(rouge_l_capped(pred, ref))
        n_pred = len(pred.split())
        lead = " ".join(row["document"].split()[:max(n_pred, 1)])
        r1_lead.append(summ_utils.rouge1_f1(lead, ref))
        rep.append(summ_utils.repetition_rate(pred))
        ratio.append(n_pred / max(len(ref.split()), 1))

    VAL_RESULT = {
        "docs": len(VAL_ROWS),
        "rouge1_f1": float(np.mean(r1)),
        "rougeL_f1_capped": float(np.mean(rl)),
        "lead_same_length_rouge1_f1": float(np.mean(r1_lead)),
        "repetition_rate_3gram": float(np.mean(rep)),
        "pred_to_ref_length_ratio": float(np.mean(ratio)),
        "pred_words_mean": float(np.mean([len(p.split()) for p in val_summaries])),
        "ref_words_mean": float(np.mean([len(r["summary"].split()) for r in VAL_ROWS])),
    }
    print("\nvalidation (my ROUGE implementation, word level; not the official scorer):")
    for key in VAL_RESULT:
        print("  %-28s %.4f" % (key, VAL_RESULT[key]))

    if USE_BERTSCORE:
        try:
            from bert_score import score as bert_score_fn
            P, R, F1 = bert_score_fn(val_summaries, [r["summary"] for r in VAL_ROWS], lang="en", verbose=False)
            VAL_RESULT["bertscore_f1"] = float(F1.mean())
            print("  bertscore_f1                 %.4f" % VAL_RESULT["bertscore_f1"])
        except Exception as e:
            print("BERTScore skipped:", repr(e))

    with open(os.path.join(OUT_DIR, "val_predictions.json"), "w", encoding="utf-8") as f:
        json.dump([{"id": r["id"], "prediction": p, "reference": r["summary"]} for r, p in zip(VAL_ROWS, val_summaries)],
                  f, ensure_ascii=False, indent=1)
    print("saved val_predictions.json")
else:
    print("Validation skipped.")

## 10. Test predictions and submission

In [ ]:
SUBMISSION_PATH = os.path.join(OUT_DIR, "prediction.json")

if STOPPED_EARLY:
    print("Training stopped early, skipping test predictions. Run again to resume.")
elif RUN_TEST and len(TEST_ROWS) > 0:
    test_summaries = summarize_documents(TEST_ROWS, "test")
    summaries = {}
    for row, text in zip(TEST_ROWS, test_summaries):
        summaries[row["id"]] = text
    sub = summ_utils.write_submission(TEST_ROWS, summaries, SUBMISSION_PATH)

    empty = sum(1 for s in sub if s["summary"].strip() == "")
    n_fallback = sum(1 for row in TEST_ROWS if summaries[row["id"]].strip() == "")
    print("summaries:", len(sub), "| empty generations replaced by the document start:", n_fallback)
    print("words per summary: mean %.0f | min %d | max %d" % (
        np.mean([len(s["summary"].split()) for s in sub]),
        min(len(s["summary"].split()) for s in sub), max(len(s["summary"].split()) for s in sub)))

    zip_path = os.path.join(OUT_DIR, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(SUBMISSION_PATH, arcname="prediction.json")
    print("wrote", zip_path)
else:
    print("Test predictions skipped (no test docs yet, or RUN_TEST is off).")

## 11. Run summary (for the report)

In [ ]:
summary = {
    "model": "Phase-1 Model 1 (GQA) + LoRMA (" + LORMA_VARIANT + ")",
    "strategy": "chunk-level summarization (sentence-aligned chunks), pieces joined in order",
    "lorma": {"variant": LORMA_VARIANT, "r": LORMA_R, "alpha": LORMA_ALPHA, "dropout": LORMA_DROPOUT, "target_layers": len(LORMA_TARGETS)},
    "params_total": int(N_BASE),
    "params_trainable": int(N_LORMA),
    "train_examples": len(PAIRS),
    "epochs": EPOCHS,
    "train_minutes": TRAIN_SECONDS / 60.0,
    "final_train_loss": LAST_LOSS,
    "validation": VAL_RESULT,
    "generation_tokens_per_second": GEN_STATS["tokens"] / max(GEN_STATS["seconds"], 1e-6),
    "peak_gpu_memory_gb": (torch.cuda.max_memory_allocated() / 1e9) if DEVICE.type == "cuda" else None,
    "stopped_early": STOPPED_EARLY,
    "chunk_doc_len": CHUNK_DOC_LEN,
    "chunk_sum_budget": CHUNK_SUM_BUDGET,
    "len_mult": LEN_MULT,
    "smoke_test": SMOKE_TEST,
}
with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
    json.dump(summary, f, indent=1, default=str)
print(json.dumps(summary, indent=1, default=str))

print("\nfiles in", OUT_DIR)
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print("  ", name, round(os.path.getsize(full) / 1e6, 2), "MB")